In [41]:
import librosa
import numpy as np
import pickle
import operator
from collections import Counter

In [42]:
def nearestClass(neighbors):
    classVote = {}

    for x in range(len(neighbors)):
        response = neighbors[x]
        if response in classVote:
            classVote[response]+=1
        else:
            classVote[response]=1

    sorter = sorted(classVote.items(), key = operator.itemgetter(1), reverse=True)
    genre_map = {
        1: 'blues',
        2: 'classical',
        3: 'country',
        4: 'disco',
        5: 'hiphop',
        6: 'jazz',
        7: 'metal',
        8: 'pop',
        9: 'reggae',
        10: 'rock'
    }
    return genre_map[sorter[0][0]]

In [43]:
import numpy as np

def distance(instance1, instance2, k):
    mm1 = instance1[0]
    cm1 = instance1[1]
    mm2 = instance2[0]
    cm2 = instance2[1]

    # Bhattacharyya distance formula
    dist = np.trace(np.dot(np.linalg.inv(cm2), cm1))
    dist += np.dot(np.dot((mm2 - mm1).T, np.linalg.inv(cm2)), (mm2 - mm1))
    dist += np.log(np.linalg.det(cm2)) - np.log(np.linalg.det(cm1))
    dist -= k
    return dist

In [44]:
# def getNeighbors(trainingSet, instance, k):
#     distances = []
#     for x in range (len(trainingSet)):
#         dist = distance(trainingSet[x], instance, k )+ distance(instance, trainingSet[x], k)
#         distances.append((trainingSet[x][2], dist))
#     distances.sort(key=operator.itemgetter(1))
#     neighbors = []
#     for x in range(k):
#         neighbors.append(distances[x][0])
#     return neighbors

In [45]:
import operator

In [46]:
def getNeighbors(X_trainingSet,Y_trainingSet, test_instance, k):
    distances=[]
    for i in range(len(X_trainingSet)):
        dist = distance(X_trainingSet[i], test_instance, k)+ distance(test_instance, X_trainingSet[i], k)
        distances.append((Y_trainingSet[i],dist))
    distances.sort(key=operator.itemgetter(1))
    neighbors = []
    for x in range(k):
        neighbors.append(distances[x][0])
    return neighbors 

In [47]:
# # STEP 2: Define KNN functions
# def euclidean_distance(a, b):
#     return np.linalg.norm(a - b)

# def getNeighbors(X_train, y_train, test_instance, k):
#     distances = []
#     for i in range(len(X_train)):
#         dist = euclidean_distance(X_train[i], test_instance)
#         distances.append((y_train[i], dist))
#     distances.sort(key=lambda x: x[1])
#     neighbors = [distances[i][0] for i in range(k)]
#     return neighbors

# def nearestClass(neighbors):
#     vote_result = Counter(neighbors)
#     return vote_result.most_common(1)[0][0]

In [48]:
# STEP 3: Load the training dataset from my.dat
dataset = []
with open("/kaggle/input/myydata/myy.dat", "rb") as f:
    while True:
        try:
            dataset.append(pickle.load(f))
        except EOFError:
            break

X_train = [x[0:2] for x in dataset]  # Feature vector (e.g., MFCC mean)
y_train = [x[2] for x in dataset]  # Genre label

In [49]:
# STEP 4: Load your new audio file and extract features
audio_path = "/kaggle/input/gtzan-dataset-music-genre-classification/Data/genres_original/metal/metal.00000.wav"  # Update this if your file has a different name

y, sr = librosa.load(audio_path, sr=None)
mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
mfcc_mean = np.mean(mfcc, axis=1)  # Extract MFCC mean as feature vector
mfcc_cov = np.cov(mfcc)

In [50]:
X_train[0]

(array([ 82.76336286, -10.55934444,   7.17268919,  -6.51186271,
         -1.73287345,  -6.03163675,   3.63545541,  -3.68575569,
         -5.77003603,   4.82542453,  -1.44515419,  -3.38199991,
          0.3057755 ]),
 array([[ 37.37609667, -22.88172046, -32.06156211, -12.16085936,
          -4.50186772, -12.06110003,  -1.85884477,  -8.32030544,
          -8.55862294,  21.11899249,   8.16767784,  -5.84774305,
          -7.647389  ],
        [-22.88172046,  73.35318071, -27.17024961, -29.40709997,
           0.94853552, -27.04308125,  31.05784542,   7.80173647,
          -8.47583805,  20.36508533,  10.5484723 ,   4.91225633,
           3.30425355],
        [-32.06156211, -27.17024961, 149.19684466,  36.91029323,
          -6.52032083,  53.70713867, -30.28228848,  20.2821179 ,
          34.06570617, -71.96655968, -22.81515908,  15.69512841,
          -2.08783985],
        [-12.16085936, -29.40709997,  36.91029323, 101.17279352,
          25.45417486,  14.00760301, -30.32288621,   1.7208572

In [51]:
# STEP 5: Predict the genre using KNN
neighbors = getNeighbors(X_train, y_train, (mfcc_mean,mfcc_cov), k=5)
prediction = nearestClass(neighbors)

In [52]:
# STEP 6: Output the result
print(f"Predicted Genre: {prediction}")

Predicted Genre: rock
